In [13]:
from typing import Tuple, List

import os
import rootutils
import pandas as pd
import random

from tqdm import tqdm
import time

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## In this Notebook I will organize all COD data into single csv

In [2]:
from rdkit import Chem
import pandas as pd
import pubchempy as pcp

from src.utils import create_df
import random
from src.cif_utils import get_canonical_smiles, get_smiles_from_chemical_name, read_cif, get_smiles_from_id_on_web

In [25]:
import numpy as np

cifs_path = "/Users/aleksandr.varlamov/cif/all_cifs"
cifs = sorted(os.listdir(cifs_path))
cifs_id = np.array(list(map(lambda x: x.split(".")[0], cifs)))
np.save("cifs_id.npy", cifs_id)

In [26]:
np.load("cifs_id.npy")

array(['1000000', '1000001', '1000002', ..., '9017924', '9017925',
       '9017926'], dtype='<U7')

In [22]:
for id in cifs_id:
    print(id)
    if int(id) in pd.read_csv("cod_parsed.csv")["id"].to_list():
        print(id)


1000000


FileNotFoundError: [Errno 2] No such file or directory: 'cod_parsed.csv'

In [11]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_2) AppleWebKit/537.36 (KHTML, like Gecko) Firefox/116.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    # Add more user agents if needed
]

In [24]:
if __name__ == "__main__":
    cifs_path = "/Users/aleksandr.varlamov/cif/all_cifs"
    cifs = sorted(os.listdir(cifs_path))
    cifs_id = list(map(lambda x: x.split(".")[0], cifs))
    csv_path = "cod_parsed.csv"

    if os.path.exists(csv_path):
        results_df = pd.read_csv(csv_path)
    else:
        results_df = pd.DataFrame(columns=["id", "smiles", "common_name"])

    for i, id in tqdm(enumerate(cifs_id), total=len(cifs_id)):
        if id in results_df['id'].values:
            continue

        try:
            smiles, common_name = get_smiles_from_id_on_web(id)
            if smiles is None and common_name is not None:
                try:
                    smiles = get_smiles_from_chemical_name(common_name)
                    print(f"Successfully got smiles from name for {id}")
                except Exception as e:
                    print(f"Failed to get smiles from name for {id}: {e}")
                    continue
                
            new_row = pd.DataFrame({"id": [id], "smiles": [smiles], "common_name": [common_name]})
            results_df = pd.concat([results_df, new_row], ignore_index=True)
            results_df.to_csv(csv_path, index=False)
            time.sleep(random.uniform(0.2, 0.5))  # Random sleep between 0.5 and 2.5 seconds

        except Exception as e:
            print(f"Failed for {id}: {e}")
            time.sleep(random.uniform(2, 5))  # Sleep longer after a failure
            continue


  0%|          | 3/523999 [00:03<183:17:56,  1.26s/it]

Failed to get smiles from name for 1000002: Compound not found: Deuterated acid strontium oxalate


  0%|          | 4/523999 [00:04<184:25:40,  1.27s/it]

Failed to get smiles from name for 1000003: Compound not found: Anhydrous acid strontium oxalate


  0%|          | 6/523999 [00:06<148:57:00,  1.02s/it]

Successfully got smiles from name for 1000006


  0%|          | 24/523999 [00:20<99:04:52,  1.47it/s] 

Successfully got smiles from name for 1000024


  0%|          | 30/523999 [00:25<124:51:12,  1.17it/s]


KeyboardInterrupt: 